# 03 — Exploración descriptiva del panel

Caracteriza los grupos sociales del panel (quintil de ingreso per cápita, condición de
actividad del jefe/a) y mide movimientos entre quintiles y en un indicador habitacional
tipo NBI. **Todo descriptivo -- nada se interpreta como causal.** Insumo para decidir la
etiqueta de deterioro más adelante, no la etiqueta en sí.

Lee `data/02_nucleo/`, `data/03_panel/` y (para `NIVEL_ED`, todavía no incorporado a
`ESQUEMA_NUCLEO` en una corrida real de 01b) `data/01_armonizado/individual/`. No
reprocesa ni re-tipa nada.

`cargar_hogar`/`cargar_individual` se importan de `pipeline_meta` (uso general,
compartido con 04/05/06) en vez de definirse acá -- quedan locales en esta notebook
`COLUMNAS_HOGAR`/`COLUMNAS_INDIVIDUAL` (qué columnas pedir) y `NUCLEO`/`ARMONIZADO`
(de qué capa leer).

**Grupo social principal**: quintil de ingreso per cápita = decil INDEC (`DECCFR`)
agrupado de a dos (1-2→Q1 … 9-10→Q5). Aparte, sin quintil, en dos categorías con
significado distinto (D3, `tipado_nucleo.ESQUEMA_NUCLEO`): `ingreso_no_declarado`
(`DECCFR==12` o `PONDIH==0`, solo observable desde 2016) y `decil_0` (`DECCFR==0`,
"ingreso cero real" -- existe en toda la serie, 11.074 casos confirmados).

**Grupo social secundario**: condición de actividad del jefe/a (ocupado/desocupado/
inactivo). Jefe ambiguo o indeterminable (0 o más de 1 persona con `CH03==1`) queda
excluido y contado aparte.

**Ponderación**: `PONDERA` para todo lo que no es ingreso (personas: pirámide/edad/
cajas; hogares: tamaño, jefatura, vivienda -- usando el `PONDERA` del jefe/a como proxy
del hogar, en toda la serie). `PONDIH` solo para estadísticas de ingreso (montos/
distribución de IPCF), desde 2016 -- antes, esas estadísticas también usan `PONDERA`
del jefe/a por no existir ponderador de ingreso histórico.

Códigos especiales (`ESTADO==0`, jefe ambiguo, `DECCFR` 0/12, valores no parseables) se
excluyen de cada estadística y se cuentan aparte -- nunca se imputan.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from perfil_hogar import *
from constantes import *

def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import pipeline_meta as pm
from armonizacion import es_historico

NUCLEO = REPO / "data" / "02_nucleo"
ARMONIZADO = REPO / "data" / "01_armonizado"
PANEL = REPO / "data" / "03_panel"
META = REPO / "data" / "meta"

# T2 de cada año 2004-2025 (2003 no tiene T2 -- la EPH continua arranca en 2003T3) +
# 2024T4 y 2025T4, para no mezclar 20 años de inflación en los gráficos de nivel.
TRIMESTRES_MUESTRA: list[tuple[int, int]] = [(anio, 2) for anio in range(2004, 2026)] + [(2024, 4), (2025, 4)]
print(f"{len(TRIMESTRES_MUESTRA)} trimestres de muestra: {TRIMESTRES_MUESTRA[0]} .. {TRIMESTRES_MUESTRA[-1]}")

In [ ]:
COLUMNAS_HOGAR = [
    "CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", "DECCFR", "PONDIH", "IPCF",
    "II1", "II2", "II5", "II5_1", "IV1", "IV3", "IV8", "IV10", "IV12_1", "IV12_2", "IV12_3",
    "V13", "V14", "V15", "V16", "V17", "ingreso_no_declarado",
]
COLUMNAS_INDIVIDUAL = [
    "CODUSU", "NRO_HOGAR", "COMPONENTE", "ANO4", "TRIMESTRE",
    "CH03", "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H", "PONDERA",
]

In [ ]:
def calcular_info_jefe(individual: pd.DataFrame) -> pd.DataFrame:
    """Por hogar: condición de actividad, sexo, `NIVEL_ED` y `PONDERA` del jefe/a
    (`PONDERA` del jefe/a se usa como proxy del peso del hogar, ver nota de arriba).
    `jefe_ambiguo=True` si hay 0 o más de 1 persona con `CH03==1` en el hogar."""
    jefe = individual[individual["CH03"] == 1]
    conteo = jefe.groupby(["CODUSU", "NRO_HOGAR"]).size()
    jefe_unico = jefe.drop_duplicates(["CODUSU", "NRO_HOGAR"]).set_index(["CODUSU", "NRO_HOGAR"])
    estado = pd.to_numeric(jefe_unico["ESTADO"], errors="coerce")
    condicion = pd.Series(pd.array([pd.NA] * len(jefe_unico), dtype="object"), index=jefe_unico.index)
    condicion[estado == 1] = "ocupado"
    condicion[estado == 2] = "desocupado"
    condicion[estado == 3] = "inactivo"
    resultado = pd.DataFrame({
        "condicion_actividad_jefe": condicion,
        "pondera_jefe": jefe_unico["PONDERA"],
        "nivel_ed_jefe": jefe_unico["NIVEL_ED"],
        "sexo_jefe": jefe_unico["CH04"],
    }).reset_index()
    resultado = resultado.merge(conteo.rename("n_jefe").reset_index(), on=["CODUSU", "NRO_HOGAR"], how="left")
    resultado["jefe_ambiguo"] = resultado["n_jefe"].fillna(0).ne(1)
    resultado = resultado.drop(columns=["n_jefe"])
    resultado.loc[resultado["jefe_ambiguo"], ["condicion_actividad_jefe", "sexo_jefe"]] = pd.NA
    return resultado

def percentiles_ponderados(valores: np.ndarray, pesos: np.ndarray, qs: list[float]) -> np.ndarray:
    """Percentiles ponderados (interpolación sobre la CDF ponderada) -- `np.percentile`
    no soporta pesos; hace falta para los boxplots ponderados por PONDERA/PONDIH."""
    orden = np.argsort(valores)
    v, w = np.asarray(valores, dtype=float)[orden], np.asarray(pesos, dtype=float)[orden]
    cw = np.cumsum(w) - 0.5 * w
    cw /= w.sum()
    return np.interp(qs, cw, v)

def stats_boxplot_ponderado(valores: np.ndarray, pesos: np.ndarray, etiqueta: str) -> dict:
    """Stats compatibles con `Axes.bxp` (boxplot ponderado manual, matplotlib no trae
    uno nativo)."""
    if len(valores) == 0:
        return {"label": etiqueta, "med": np.nan, "q1": np.nan, "q3": np.nan,
                "whislo": np.nan, "whishi": np.nan, "fliers": []}
    q1, mediana, q3 = percentiles_ponderados(valores, pesos, [0.25, 0.5, 0.75])
    riq = q3 - q1
    whislo = max(valores.min(), q1 - 1.5 * riq)
    whishi = min(valores.max(), q3 + 1.5 * riq)
    return {"label": etiqueta, "med": mediana, "q1": q1, "q3": q3,
            "whislo": whislo, "whishi": whishi, "fliers": []}

def calcular_estrategias(hogar: pd.DataFrame) -> pd.DataFrame:
    v = {c: pd.to_numeric(hogar[c], errors="coerce") for c in ("V13", "V14", "V15", "V16", "V17")}
    recurre_desahorro = (v["V13"] == 1) | (v["V14"] == 1) | (v["V17"] == 1)
    recurre_endeudamiento_formal = (v["V15"] == 1) | (v["V16"] == 1)
    return pd.DataFrame({"recurre_desahorro": recurre_desahorro,
                          "recurre_endeudamiento_formal": recurre_endeudamiento_formal}, index=hogar.index)

## Parte A — Gráficos elementales por grupo social

Se arma un DataFrame persona-trimestre para `TRIMESTRES_MUESTRA`, con el quintil y la
condición de actividad del jefe/a ya asignados a cada persona de su hogar.

In [ ]:
partes_personas = []
filas_excluidos_a = []

for anio, trimestre in TRIMESTRES_MUESTRA:
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR, NUCLEO)
    individual = pm.cargar_individual(anio, trimestre, COLUMNAS_INDIVIDUAL, NUCLEO, ARMONIZADO)
    if hogar is None or individual is None:
        continue

    hogar = hogar.copy()
    hogar["grupo_ingreso"] = calcular_grupo_ingreso(hogar)
    info_jefe = calcular_info_jefe(individual)
    hogar = hogar.merge(info_jefe, on=["CODUSU", "NRO_HOGAR"], how="left")
    hogar["jefe_ambiguo"] = hogar["jefe_ambiguo"].fillna(True)

    personas = individual.merge(
        hogar[["CODUSU", "NRO_HOGAR", "grupo_ingreso", "condicion_actividad_jefe",
               "pondera_jefe", "nivel_ed_jefe", "jefe_ambiguo"]],
        on=["CODUSU", "NRO_HOGAR"], how="left",
    )
    personas["ANO4"] = anio
    personas["TRIMESTRE"] = trimestre
    partes_personas.append(personas)

    n_hogares = len(hogar)
    filas_excluidos_a.append({
        "anio": anio, "trimestre": trimestre, "n_hogares": n_hogares,
        "n_sin_quintil_q1_q5": int((~hogar["grupo_ingreso"].isin(ORDEN_QUINTILES)).sum()),
        "n_jefe_ambiguo_o_indeterminable": int(hogar["jefe_ambiguo"].sum()),
    })

personas_muestra = pd.concat(partes_personas, ignore_index=True)
excluidos_a = pd.DataFrame(filas_excluidos_a)
excluidos_a.to_csv(META / "exploracion_excluidos_parte_a.csv", index=False)
print(f"{len(personas_muestra):,} personas en la muestra, {len(TRIMESTRES_MUESTRA)} trimestres")
display(excluidos_a)

### 1. Pirámide de población por quintil

Sexo × grupo de edad quinquenal, ponderado por `PONDERA`. Excluye personas sin quintil
Q1-Q5 (decil 0, ingreso no declarado, DECCFR no parseable).

In [ ]:
BINS_EDAD = list(range(0, 101, 5))
ETIQUETAS_EDAD = [f"{i}-{i+4}" for i in BINS_EDAD[:-1]]

piramide_base = personas_muestra[personas_muestra["grupo_ingreso"].isin(ORDEN_QUINTILES) & personas_muestra["CH06"].notna() & personas_muestra["CH04"].notna()].copy()
piramide_base["grupo_edad"] = pd.cut(piramide_base["CH06"], bins=BINS_EDAD + [200], labels=ETIQUETAS_EDAD + ["100+"], right=False)

tabla_piramide = (
    piramide_base.groupby(["grupo_ingreso", "grupo_edad", "CH04"], observed=True)["PONDERA"]
    .sum().reset_index()
)
tabla_piramide.to_csv(META / "exploracion_piramide_poblacion.csv", index=False)

fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)
for ax, q in zip(axes, ORDEN_QUINTILES):
    sub = tabla_piramide[tabla_piramide["grupo_ingreso"] == q]
    varones = sub[sub["CH04"] == 1].set_index("grupo_edad")["PONDERA"].reindex(ETIQUETAS_EDAD + ["100+"], fill_value=0)
    mujeres = sub[sub["CH04"] == 2].set_index("grupo_edad")["PONDERA"].reindex(ETIQUETAS_EDAD + ["100+"], fill_value=0)
    ax.barh(varones.index, -varones.values, color="steelblue", label="Varones")
    ax.barh(mujeres.index, mujeres.values, color="indianred", label="Mujeres")
    ax.set_title(q)
    ax.axvline(0, color="black", linewidth=0.5)
axes[0].set_ylabel("Grupo de edad")
axes[-1].legend(loc="lower right")
fig.suptitle("Pirámide de población por quintil de ingreso per cápita del hogar")
plt.tight_layout()
plt.show()
display(tabla_piramide)

### 2. Histograma de edad por sexo, por quintil

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
tabla_hist_edad = []
for ax, q in zip(axes, ORDEN_QUINTILES):
    sub = piramide_base[piramide_base["grupo_ingreso"] == q]
    for sexo, color, nombre in [(1, "steelblue", "Varones"), (2, "indianred", "Mujeres")]:
        s = sub[sub["CH04"] == sexo]
        ax.hist(s["CH06"], bins=20, weights=s["PONDERA"], alpha=0.5, color=color, label=nombre)
        tabla_hist_edad.append({"grupo_ingreso": q, "sexo": nombre, "edad_media_ponderada": np.average(s["CH06"], weights=s["PONDERA"]) if len(s) else np.nan})
    ax.set_title(q)
    ax.set_xlabel("Edad")
axes[0].set_ylabel("Población ponderada")
axes[0].legend()
fig.suptitle("Distribución de edad por sexo, por quintil")
plt.tight_layout()
plt.show()
tabla_hist_edad = pd.DataFrame(tabla_hist_edad)
tabla_hist_edad.to_csv(META / "exploracion_edad_media_por_quintil_sexo.csv", index=False)
display(tabla_hist_edad)

### 3. Diagrama de caja de edad por quintil y sexo

In [ ]:
stats_caja_edad = []
for q in ORDEN_QUINTILES:
    for sexo, nombre in [(1, "Varones"), (2, "Mujeres")]:
        s = piramide_base[(piramide_base["grupo_ingreso"] == q) & (piramide_base["CH04"] == sexo)]
        stats_caja_edad.append(stats_boxplot_ponderado(s["CH06"].to_numpy(), s["PONDERA"].to_numpy(), f"{q}\n{nombre}"))

fig, ax = plt.subplots(figsize=(14, 5))
ax.bxp(stats_caja_edad, showfliers=False)
ax.set_ylabel("Edad")
ax.set_title("Edad por quintil y sexo (ponderado por PONDERA)")
plt.tight_layout()
plt.show()
tabla_caja_edad = pd.DataFrame(stats_caja_edad)
tabla_caja_edad.to_csv(META / "exploracion_caja_edad_quintil_sexo.csv", index=False)
display(tabla_caja_edad)

### 4. Tamaño del hogar por quintil

A nivel hogar (no persona) -- cantidad de integrantes, ponderado por `PONDERA` del
jefe/a.

In [ ]:
tamano_hogar = personas_muestra.groupby(["ANO4", "TRIMESTRE", "CODUSU", "NRO_HOGAR", "grupo_ingreso", "pondera_jefe"], observed=True).size().reset_index(name="n_miembros")
tamano_hogar = tamano_hogar[tamano_hogar["grupo_ingreso"].isin(ORDEN_QUINTILES)]

stats_caja_tamano = []
for q in ORDEN_QUINTILES:
    s = tamano_hogar[tamano_hogar["grupo_ingreso"] == q]
    stats_caja_tamano.append(stats_boxplot_ponderado(s["n_miembros"].to_numpy(), s["pondera_jefe"].fillna(1).to_numpy(), q))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bxp(stats_caja_tamano, showfliers=False)
ax.set_ylabel("Cantidad de integrantes del hogar")
ax.set_title("Tamaño del hogar por quintil (ponderado por PONDERA del jefe/a)")
plt.tight_layout()
plt.show()
tabla_caja_tamano = pd.DataFrame(stats_caja_tamano)
tabla_caja_tamano.to_csv(META / "exploracion_tamano_hogar_quintil.csv", index=False)
display(tabla_caja_tamano)

### 5. % de mujeres jefas de hogar, por quintil

In [ ]:
hogares_unicos = personas_muestra.drop_duplicates(["ANO4", "TRIMESTRE", "CODUSU", "NRO_HOGAR"])[
    ["ANO4", "TRIMESTRE", "CODUSU", "NRO_HOGAR", "grupo_ingreso", "pondera_jefe"]
].merge(
    personas_muestra[personas_muestra["CH03"] == 1][["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "CH04"]].drop_duplicates(["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE"]),
    on=["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE"], how="left",
)
hogares_unicos = hogares_unicos[hogares_unicos["grupo_ingreso"].isin(ORDEN_QUINTILES) & hogares_unicos["CH04"].notna()]

tabla_jefatura = hogares_unicos.groupby("grupo_ingreso").apply(
    lambda g: pd.Series({
        "pct_mujeres_jefas": np.average(g["CH04"] == 2, weights=g["pondera_jefe"].fillna(1)) * 100,
        "n_hogares": len(g),
    }), include_groups=False,
).reindex(ORDEN_QUINTILES)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(tabla_jefatura.index, tabla_jefatura["pct_mujeres_jefas"], color="indianred")
ax.set_ylabel("% de hogares con jefa mujer")
ax.set_title("Jefatura femenina por quintil (ponderado por PONDERA del jefe/a)")
plt.tight_layout()
plt.show()
tabla_jefatura.to_csv(META / "exploracion_jefatura_femenina_quintil.csv")
display(tabla_jefatura)

### 6. IPCF (escala log) por quintil y por condición de actividad del jefe/a

Ponderado por `PONDIH` desde 2016, por `PONDERA` del jefe/a antes (única estadística de
ingreso de esta parte). Corrobora que los quintiles separan bien.

In [ ]:
hogares_ipcf = personas_muestra.drop_duplicates(["ANO4", "TRIMESTRE", "CODUSU", "NRO_HOGAR"])[
    ["ANO4", "TRIMESTRE", "CODUSU", "NRO_HOGAR", "grupo_ingreso", "condicion_actividad_jefe", "pondera_jefe"]
]
hogar_ipcf_partes = []
for anio, trimestre in TRIMESTRES_MUESTRA:
    h = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR, NUCLEO)
    if h is None:
        continue
    h = h.reindex(columns=["CODUSU", "NRO_HOGAR", "IPCF", "PONDIH"]).copy()
    h["ANO4"], h["TRIMESTRE"] = anio, trimestre
    hogar_ipcf_partes.append(h)
hogar_ipcf = pd.concat(hogar_ipcf_partes, ignore_index=True)

hogares_ipcf = hogares_ipcf.merge(hogar_ipcf, on=["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE"], how="left")
hogares_ipcf["peso_ingreso"] = hogares_ipcf["PONDIH"].where(hogares_ipcf["PONDIH"] > 0, hogares_ipcf["pondera_jefe"])
hogares_ipcf = hogares_ipcf[hogares_ipcf["IPCF"] > 0]  # log requiere > 0; IPCF=0 real queda excluido de este gráfico puntual

stats_caja_ipcf_q = []
for q in ORDEN_QUINTILES:
    s = hogares_ipcf[hogares_ipcf["grupo_ingreso"] == q]
    stats_caja_ipcf_q.append(stats_boxplot_ponderado(np.log10(s["IPCF"].to_numpy()), s["peso_ingreso"].fillna(1).to_numpy(), q))

stats_caja_ipcf_cond = []
for cond in ["ocupado", "desocupado", "inactivo"]:
    s = hogares_ipcf[hogares_ipcf["condicion_actividad_jefe"] == cond]
    stats_caja_ipcf_cond.append(stats_boxplot_ponderado(np.log10(s["IPCF"].to_numpy()), s["peso_ingreso"].fillna(1).to_numpy(), cond))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bxp(stats_caja_ipcf_q, showfliers=False)
axes[0].set_ylabel("log10(IPCF)")
axes[0].set_title("IPCF por quintil")
axes[1].bxp(stats_caja_ipcf_cond, showfliers=False)
axes[1].set_title("IPCF por condición de actividad del jefe/a")
plt.tight_layout()
plt.show()

tabla_caja_ipcf = pd.concat([pd.DataFrame(stats_caja_ipcf_q).assign(eje="quintil"),
                              pd.DataFrame(stats_caja_ipcf_cond).assign(eje="condicion_jefe")], ignore_index=True)
tabla_caja_ipcf.to_csv(META / "exploracion_ipcf_log_quintil_condicion.csv", index=False)
display(tabla_caja_ipcf)

## Parte B — Cantidades y movimientos entre quintiles

### 1. Cantidades por trimestre y quintil

In [ ]:
filas_cantidades = []
for anio, trimestre in TRIMESTRES_MUESTRA:
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR, NUCLEO)
    individual = pm.cargar_individual(anio, trimestre, COLUMNAS_INDIVIDUAL, NUCLEO, ARMONIZADO)
    if hogar is None or individual is None:
        continue
    hogar = hogar.copy()
    hogar["grupo_ingreso"] = calcular_grupo_ingreso(hogar)
    info_jefe = calcular_info_jefe(individual)
    hogar = hogar.merge(info_jefe[["CODUSU", "NRO_HOGAR", "pondera_jefe"]], on=["CODUSU", "NRO_HOGAR"], how="left")
    n_miembros = individual.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros")
    suma_pondera_personas = individual.groupby(["CODUSU", "NRO_HOGAR"])["PONDERA"].sum().rename("suma_pondera_personas")
    hogar = hogar.merge(n_miembros, on=["CODUSU", "NRO_HOGAR"], how="left")
    hogar = hogar.merge(suma_pondera_personas, on=["CODUSU", "NRO_HOGAR"], how="left")

    for grupo, sub in hogar.groupby(hogar["grupo_ingreso"].fillna("sin_dato_deccfr")):
        filas_cantidades.append({
            "anio": anio, "trimestre": trimestre, "grupo_ingreso": grupo,
            "n_hogares": len(sub), "n_personas": int(sub["n_miembros"].sum()),
            "poblacion_hogares": float(sub["pondera_jefe"].fillna(0).sum()),
            "poblacion_personas": float(sub["suma_pondera_personas"].fillna(0).sum()),
        })

quintiles_cantidades = pd.DataFrame(filas_cantidades)
total_personas_trim = quintiles_cantidades.groupby(["anio", "trimestre"])["poblacion_personas"].transform("sum")
quintiles_cantidades["pct_poblacion_personas"] = quintiles_cantidades["poblacion_personas"] / total_personas_trim * 100
quintiles_cantidades.to_csv(META / "quintiles_cantidades.csv", index=False)

print("--- % de personas ponderadas por quintil, promedio en toda la muestra ---")
print("(no se espera ~20% parejo -- DECCFR reparte hogares, no personas; los hogares")
print(" pobres son más grandes, sobrerrepresentan personas en Q1)")
print(quintiles_cantidades[quintiles_cantidades["grupo_ingreso"].isin(ORDEN_QUINTILES)]
      .groupby("grupo_ingreso")["pct_poblacion_personas"].mean().reindex(ORDEN_QUINTILES).round(1))
print()
print("--- hogares excluidos por trimestre (decil_0 + ingreso_no_declarado), últimos 5 ---")
display(quintiles_cantidades[quintiles_cantidades["grupo_ingreso"].isin(["decil_0", "ingreso_no_declarado"])].tail(10))

**Nota metodológica**: el quintil es una posición *relativa* de hogares según su
ingreso per cápita (D9/D10-style ranking), no un tramo de 20% de personas ni de ingreso
real. Moverse de quintil entre `t` y `t+h` no equivale a ganar o perder ingreso real --
puede reflejar que otros hogares se movieron, o simple ruido de medición (punto 4).

### 2-3. Movimientos entre quintiles (t→t+1, t→t+4), por era y por mandato

Usa el panel completo (todos los trimestres publicados, no solo `TRIMESTRES_MUESTRA` --
el quintil es una posición relativa, no un nivel monetario, así que no hay "mezcla de
inflación" que evitar acá). Filtra a `identidad_confirmada == True` y quintil Q1-Q5
conocido en ambas puntas.

In [ ]:
import panel
from armonizacion import trimestres_publicados


def cargar_deccfr(anio, trimestre):
    p = pm.ruta_particion(NUCLEO, "hogar", anio, trimestre)
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    return df.reindex(columns=["CODUSU", "NRO_HOGAR", "DECCFR", "PONDIH"])


def construir_transiciones_quintil(h):
    partes = []
    for anio_t, trimestre_t in trimestres_publicados():
        p_pares = PANEL / f"pares_h{h}" / f"ANO4={anio_t}" / f"TRIMESTRE={trimestre_t}" / "data.parquet"
        if not p_pares.exists():
            continue
        pares = pd.read_parquet(p_pares, columns=[
            "CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th", "identidad_confirmada",
            "regimen_ingreso", "mandato_presidencial_t",
        ])
        pares = pares[pares["identidad_confirmada"] == True]
        if pares.empty:
            continue
        anio_th, trimestre_th = panel.sumar_trimestres(anio_t, trimestre_t, h)
        deccfr_t = cargar_deccfr(anio_t, trimestre_t)
        deccfr_th = cargar_deccfr(anio_th, trimestre_th)
        if deccfr_t is None or deccfr_th is None:
            continue
        deccfr_t = deccfr_t.assign(grupo_ingreso_t=calcular_grupo_ingreso(deccfr_t)).rename(columns={"NRO_HOGAR": "NRO_HOGAR_t"})
        deccfr_th = deccfr_th.assign(grupo_ingreso_th=calcular_grupo_ingreso(deccfr_th)).rename(columns={"NRO_HOGAR": "NRO_HOGAR_th"})
        pares = pares.merge(deccfr_t[["CODUSU", "NRO_HOGAR_t", "grupo_ingreso_t"]], on=["CODUSU", "NRO_HOGAR_t"], how="left")
        pares = pares.merge(deccfr_th[["CODUSU", "NRO_HOGAR_th", "grupo_ingreso_th"]], on=["CODUSU", "NRO_HOGAR_th"], how="left")
        pares = pares[pares["grupo_ingreso_t"].isin(ORDEN_QUINTILES) & pares["grupo_ingreso_th"].isin(ORDEN_QUINTILES)]
        if pares.empty:
            continue
        pares["h"] = h
        partes.append(pares[["h", "regimen_ingreso", "mandato_presidencial_t", "grupo_ingreso_t", "grupo_ingreso_th"]])
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(
        columns=["h", "regimen_ingreso", "mandato_presidencial_t", "grupo_ingreso_t", "grupo_ingreso_th"])


t0 = time.time()
transiciones = pd.concat([construir_transiciones_quintil(1), construir_transiciones_quintil(4)], ignore_index=True)
print(f"{len(transiciones):,} pares con identidad confirmada y quintil conocido en ambas puntas ({time.time()-t0:.1f}s)")

In [ ]:
def resumen_transicion(df, nombre):
    matriz_n = pd.crosstab(df["grupo_ingreso_t"], df["grupo_ingreso_th"]).reindex(index=ORDEN_QUINTILES, columns=ORDEN_QUINTILES, fill_value=0)
    matriz_pct = (matriz_n.div(matriz_n.sum(axis=1), axis=0) * 100).round(1)
    escalon = df["grupo_ingreso_th"].str[1].astype(int) - df["grupo_ingreso_t"].str[1].astype(int)
    resumen = pd.Series({
        "n_pares": len(df),
        "pct_mismo_quintil": (escalon == 0).mean() * 100,
        "pct_sube": (escalon > 0).mean() * 100,
        "pct_baja": (escalon < 0).mean() * 100,
        "escalones_promedio_si_sube": escalon[escalon > 0].mean() if (escalon > 0).any() else np.nan,
        "escalones_promedio_si_baja": escalon[escalon < 0].mean() if (escalon < 0).any() else np.nan,
    })
    print(f"--- {nombre} (n={len(df):,}) ---")
    print(f"mismo quintil: {resumen['pct_mismo_quintil']:.1f}%  sube: {resumen['pct_sube']:.1f}%  baja: {resumen['pct_baja']:.1f}%")
    display(matriz_pct)
    return matriz_n, matriz_pct, resumen


filas_resumen_transicion = []
tablas_transicion = []
for h in (1, 4):
    sub_h = transiciones[transiciones["h"] == h]
    matriz_n, matriz_pct, resumen = resumen_transicion(sub_h, f"h={h}, TODOS los orígenes")
    resumen["h"], resumen["recorte"] = h, "todos"
    filas_resumen_transicion.append(resumen)
    for i_t in ORDEN_QUINTILES:
        for i_th in ORDEN_QUINTILES:
            tablas_transicion.append({"h": h, "recorte": "todos", "grupo_ingreso_t": i_t, "grupo_ingreso_th": i_th,
                                        "n": int(matriz_n.loc[i_t, i_th]), "pct_fila": float(matriz_pct.loc[i_t, i_th])})

    for era in ("historico", "regular"):
        sub = sub_h[sub_h["regimen_ingreso"] == era]
        if sub.empty:
            continue
        matriz_n, matriz_pct, resumen = resumen_transicion(sub, f"h={h}, era {era}")
        resumen["h"], resumen["recorte"] = h, f"era_{era}"
        filas_resumen_transicion.append(resumen)
        for i_t in ORDEN_QUINTILES:
            for i_th in ORDEN_QUINTILES:
                tablas_transicion.append({"h": h, "recorte": f"era_{era}", "grupo_ingreso_t": i_t, "grupo_ingreso_th": i_th,
                                            "n": int(matriz_n.loc[i_t, i_th]), "pct_fila": float(matriz_pct.loc[i_t, i_th])})

    for mandato in panel.MANDATO_PRESIDENCIAL:
        nombre_mandato = mandato[1]
        sub = sub_h[sub_h["mandato_presidencial_t"] == nombre_mandato]
        if sub.empty:
            continue
        matriz_n, matriz_pct, resumen = resumen_transicion(sub, f"h={h}, mandato {nombre_mandato}")
        resumen["h"], resumen["recorte"] = h, f"mandato_{nombre_mandato}"
        filas_resumen_transicion.append(resumen)
        for i_t in ORDEN_QUINTILES:
            for i_th in ORDEN_QUINTILES:
                tablas_transicion.append({"h": h, "recorte": f"mandato_{nombre_mandato}", "grupo_ingreso_t": i_t, "grupo_ingreso_th": i_th,
                                            "n": int(matriz_n.loc[i_t, i_th]), "pct_fila": float(matriz_pct.loc[i_t, i_th])})

resumen_transiciones_df = pd.DataFrame(filas_resumen_transicion)
resumen_transiciones_df.to_csv(META / "quintiles_transiciones_resumen.csv", index=False)
pd.DataFrame(tablas_transicion).to_csv(META / "quintiles_transiciones_matriz.csv", index=False)

**Nota (punto 5)**: las matrices de arriba varían poco entre mandatos (mismo
quintil: 52%-56% en casi todos los recortes) -- consistente con que el quintil es una
posición *relativa*: un cambio en el nivel de ingreso real de todo el país no necesita
alterar demasiado el orden relativo de los hogares entre sí. Por eso la futura etiqueta
de deterioro (fuera del alcance de esta notebook) no se basará solo en el quintil --
combinará nivel de ingreso real (deflactado) y situación laboral del hogar (Parte E),
que sí capturan cambios de nivel y no solo de posición relativa.


**Nota explícita**: parte del movimiento entre quintiles de arriba es error de
medición del ingreso (y de imputación de no-respuesta antes de 2016, D3), no movilidad
real -- no se interpreta como tal en esta notebook.

### 4. Hogares con ingreso no declarado (era regular): comparación descriptiva

Puramente descriptivo -- no se interpreta como causal ni se usa para imputar ingreso.
Compara, entre hogares con ingreso declarado y sin declarar (`ingreso_no_declarado`,
ver D2/D3, solo existe desde 2016), cuatro dimensiones no monetarias: déficit
habitacional (ancla, Parte E), nivel educativo del jefe/a, situación laboral del hogar
y aglomerado -- todas ponderadas por `PONDERA` del jefe/a (mismo criterio que el resto
de la notebook), sobre toda la era regular (2016-2025), no solo `TRIMESTRES_MUESTRA`
(acá no hay niveles monetarios de por medio, no hace falta evitar mezclar inflación).


In [ ]:
filas_ing_no_declarado = []
for anio, trimestre in trimestres_publicados():
    if es_historico(anio, trimestre):
        continue  # PONDIH e `ingreso_no_declarado` solo existen desde 2016 (D3)
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR, NUCLEO)
    individual = pm.cargar_individual(anio, trimestre, COLUMNAS_INDIVIDUAL, NUCLEO, ARMONIZADO)
    if hogar is None or individual is None:
        continue
    hogar = hogar.copy()
    n_miembros = individual.groupby(["CODUSU", "NRO_HOGAR"]).size()
    ancla = calcular_componentes_ancla(hogar.set_index(["CODUSU", "NRO_HOGAR"]), n_miembros).reset_index()
    situacion = calcular_situacion_laboral(individual)
    info_jefe = calcular_info_jefe(individual)

    hogar = hogar.merge(ancla[["CODUSU", "NRO_HOGAR", "ancla_con_deficit"]], on=["CODUSU", "NRO_HOGAR"], how="left")
    hogar = hogar.merge(situacion, on=["CODUSU", "NRO_HOGAR"], how="left")
    hogar = hogar.merge(info_jefe[["CODUSU", "NRO_HOGAR", "nivel_ed_jefe", "pondera_jefe"]], on=["CODUSU", "NRO_HOGAR"], how="left")
    hogar["nivel_ed_bucket_jefe"] = bucket_nivel_ed(hogar["nivel_ed_jefe"])
    hogar["grupo_declara_ingreso"] = hogar["ingreso_no_declarado"].map({True: "sin_declarar", False: "con_declarar"})
    filas_ing_no_declarado.append(hogar[[
        "grupo_declara_ingreso", "pondera_jefe", "AGLOMERADO",
        "ancla_con_deficit", "nivel_ed_bucket_jefe", "situacion_laboral",
    ]])

base_ing_no_declarado = pd.concat(filas_ing_no_declarado, ignore_index=True)
base_ing_no_declarado = base_ing_no_declarado[base_ing_no_declarado["grupo_declara_ingreso"].notna()]
print(f"{len(base_ing_no_declarado):,} hogares, era regular")
display(base_ing_no_declarado.groupby("grupo_declara_ingreso").size().rename("n_hogares"))


def pct_ponderado(sub, col, pesos="pondera_jefe"):
    """Distribución ponderada de `col` (usa PONDERA del jefe/a como proxy del peso
    del hogar, igual criterio que el resto de la notebook)."""
    d = sub[[col, pesos]].dropna()
    tot = d[pesos].sum()
    return (d.groupby(col)[pesos].sum() / tot * 100).round(1)


filas_comparacion = []
for grupo, sub in base_ing_no_declarado.groupby("grupo_declara_ingreso"):
    n_hogares = len(sub)
    poblacion = sub["pondera_jefe"].fillna(0).sum()
    filas_comparacion.append({"dimension": "n_hogares_y_poblacion", "categoria": "n_hogares (sin ponderar)", "grupo": grupo, "valor": n_hogares})
    filas_comparacion.append({"dimension": "n_hogares_y_poblacion", "categoria": "poblacion_hogares (ponderada)", "grupo": grupo, "valor": round(poblacion)})

    for cat, val in pct_ponderado(sub, "ancla_con_deficit").items():
        filas_comparacion.append({"dimension": "deficit_habitacional", "categoria": str(cat), "grupo": grupo, "valor": val})
    for cat, val in pct_ponderado(sub, "nivel_ed_bucket_jefe").items():
        filas_comparacion.append({"dimension": "nivel_educativo_jefe", "categoria": cat, "grupo": grupo, "valor": val})
    for cat, val in pct_ponderado(sub, "situacion_laboral").items():
        filas_comparacion.append({"dimension": "situacion_laboral_hogar", "categoria": cat, "grupo": grupo, "valor": val})
    for cat, val in pct_ponderado(sub, "AGLOMERADO").items():
        filas_comparacion.append({"dimension": "aglomerado", "categoria": str(int(cat)), "grupo": grupo, "valor": val})

comparacion_ing_no_declarado = pd.DataFrame(filas_comparacion)
comparacion_ing_no_declarado.to_csv(META / "ingreso_no_declarado_comparacion.csv", index=False)

print("--- % con déficit habitacional, por grupo (ponderado por PONDERA del jefe/a) ---")
display(comparacion_ing_no_declarado[comparacion_ing_no_declarado["dimension"] == "deficit_habitacional"]
        .pivot(index="categoria", columns="grupo", values="valor"))
print("--- Nivel educativo del jefe/a, distribución ponderada ---")
display(comparacion_ing_no_declarado[comparacion_ing_no_declarado["dimension"] == "nivel_educativo_jefe"]
        .pivot(index="categoria", columns="grupo", values="valor"))
print("--- Situación laboral del hogar, distribución ponderada ---")
display(comparacion_ing_no_declarado[comparacion_ing_no_declarado["dimension"] == "situacion_laboral_hogar"]
        .pivot(index="categoria", columns="grupo", values="valor").reindex(ORDEN_SITUACION_LABORAL))
print("--- Aglomerado: 10 con mayor diferencia absoluta entre grupos ---")
piv_agl = (comparacion_ing_no_declarado[comparacion_ing_no_declarado["dimension"] == "aglomerado"]
           .pivot(index="categoria", columns="grupo", values="valor").fillna(0))
piv_agl["diferencia_abs"] = (piv_agl["sin_declarar"] - piv_agl["con_declarar"]).abs()
display(piv_agl.sort_values("diferencia_abs", ascending=False).head(10))
print("\nTabla completa (todas las dimensiones y aglomerados) en data/meta/ingreso_no_declarado_comparacion.csv")


**Nota**: estas diferencias son asociativas, describen quiénes son los hogares que no
declaran ingreso -- no explican por qué no lo declaran ni se usan para imputarles un
ingreso.

`poblacion_hogares (ponderada)` es una suma acumulada de `PONDERA` del jefe/a a lo
largo de ~40 trimestres de la era regular -- no es un conteo de hogares distintos, un
mismo hogar real aporta a esa suma en cada trimestre en que aparece en el panel.

**Nota (punto 4) -- sesgo y su consecuencia para la etiqueta**: el hogar que no declara
ingreso es, en conjunto, *menos* vulnerable (más empleo formal e independiente, menos
sin ocupados, menos déficit habitacional, algo más de educación) y está fuertemente
sobrerrepresentado en Partidos del GBA (aglomerado 33: 50.5% de los que no declaran vs.
37.0% de los que sí declaran, +13.5 puntos -- el aglomerado con mayor diferencia,
lejos del resto). Consecuencia directa para cualquier análisis por nivel de ingreso
(quintil, IPCF) en la era regular: al quedar afuera del cálculo de quintil/IPCF, el eje
de ingreso post-2016 **sobrerrepresenta a los hogares más vulnerables** -- los hogares
menos vulnerables que no declaran ingreso no entran en esas distribuciones. Esto no se
corrige acá (no se imputa ingreso); queda documentado como limitación a tener en cuenta
si más adelante se usa el quintil o el IPCF como parte de la etiqueta de deterioro.


## Nota (D15) — vínculo con el reencuadre del TFI

El eje del TFI (D15, `docs/decisiones_metodologicas.md`) es el riesgo de un hogar de
**caer** en mayor informalidad laboral o de **no salir** de ella, por quintil de ingreso
y condiciones habitacionales -- el mandato presidencial queda como variable de contexto,
no como eje. Parte E, más abajo, ya mide las dos direcciones de forma descriptiva,
aunque fue escrita antes de D15:
- **Permanencia / no salida**: `informal_se_queda`/`informal_pasa_a_formal` (más abajo).
- **Caída**: `cae_desde_algun_formal`/`formal_pasa_a_informal` -- la definición más
  simple de "caída" de las que evalúa D15/D16 (pérdida del empleo formal del hogar).

D15 cita estos mismos números (`condiciones_vida_resumen_deficit.csv`) en su tabla de
brecha por `ancla_con_deficit`. El notebook 06 (población en riesgo) todavía no
incorpora la caída -- cuando lo haga, va a usar una definición distinta (intensificación
dentro de la informalidad, D16), no esta; esta sección sigue siendo la referencia para la
definición "pérdida del empleo formal del hogar" en particular.

## Parte E — Ancla habitacional y transiciones de condiciones de vida

Descriptivo, sin modelo -- nada se interpreta como causal.

### 1. Ancla habitacional (adaptación del criterio NBI del INDEC)

- `hacinamiento_nbi` (**usado en el ancla**): `n_miembros / II1 > 3` (criterio NBI
  estándar de personas por ambiente de uso exclusivo).
- `hacinamiento_para_dormir` (se reporta aparte, no entra en el ancla): `n_miembros /
  cuartos_para_dormir > 3`, con `cuartos_para_dormir = II2 + II5_1` (si `II5==1`).
- `calidad_materiales_deficitaria` (**adaptación**, no el criterio NBI textual
  completo): vivienda tipo inquilinato/hotel-pensión/local (`IV1` 3-5) o piso de tierra
  (`IV3==3`).
- `condiciones_sanitarias_deficitarias`: sin baño/letrina (`IV8==2`) o letrina sin
  arrastre de agua (`IV10==3`).
- `ancla_con_deficit = hacinamiento_nbi | calidad_materiales_deficitaria | condiciones_sanitarias_deficitarias`.
- `entorno_deficitario` (**aparte, no entra en el ancla**): basural, zona inundable o
  villa de emergencia (`IV12_1/2/3`).

**Qué NO se puede calcular con la EPH núcleo actual**: el NBI oficial tiene 5
dimensiones, acá se cubren 3. Faltan *inasistencia escolar* (necesita `CH10`/`CH12`,
existen en la EPH pero no están en `ESQUEMA_NUCLEO` hoy) y *capacidad de subsistencia*
(combina educación del jefe/a y tasa de dependencia -- con `NIVEL_ED` se podría
aproximar, pero no replica la fórmula censal exacta). No se implementan en esta
notebook.

### 2-3. Estabilidad del ancla, y estado de condiciones de vida + transiciones

Un solo recorrido del panel construye la base para ambos puntos (mismo patrón que
Parte B): por cada par con `identidad_confirmada`, componentes del ancla y situación
laboral en `t` y `t+h`, quintil y bucket de `NIVEL_ED` del jefe/a en `t`.

In [ ]:
def construir_base_ancla_condiciones(h):
    partes = []
    for anio_t, trimestre_t in trimestres_publicados():
        p_pares = PANEL / f"pares_h{h}" / f"ANO4={anio_t}" / f"TRIMESTRE={trimestre_t}" / "data.parquet"
        if not p_pares.exists():
            continue
        pares = pd.read_parquet(p_pares, columns=["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th", "identidad_confirmada"])
        pares = pares[pares["identidad_confirmada"] == True]
        if pares.empty:
            continue
        anio_th, trimestre_th = panel.sumar_trimestres(anio_t, trimestre_t, h)

        hogar_t = pm.cargar_hogar(anio_t, trimestre_t, COLUMNAS_HOGAR, NUCLEO)
        hogar_th = pm.cargar_hogar(anio_th, trimestre_th, COLUMNAS_HOGAR, NUCLEO)
        individual_t = pm.cargar_individual(anio_t, trimestre_t, COLUMNAS_INDIVIDUAL, NUCLEO, ARMONIZADO)
        individual_th = pm.cargar_individual(anio_th, trimestre_th, COLUMNAS_INDIVIDUAL, NUCLEO, ARMONIZADO)
        if hogar_t is None or hogar_th is None or individual_t is None or individual_th is None:
            continue

        n_t = individual_t.groupby(["CODUSU", "NRO_HOGAR"]).size()
        n_th = individual_th.groupby(["CODUSU", "NRO_HOGAR"]).size()
        ancla_t = calcular_componentes_ancla(hogar_t.set_index(["CODUSU", "NRO_HOGAR"]), n_t).reset_index()
        ancla_th = calcular_componentes_ancla(hogar_th.set_index(["CODUSU", "NRO_HOGAR"]), n_th).reset_index()
        estrategias_t = calcular_estrategias(hogar_t)
        situacion_t = calcular_situacion_laboral(individual_t)
        situacion_th = calcular_situacion_laboral(individual_th)
        info_jefe_t = calcular_info_jefe(individual_t)

        hogar_t = hogar_t.assign(grupo_ingreso=calcular_grupo_ingreso(hogar_t))
        hogar_t = hogar_t.merge(info_jefe_t[["CODUSU", "NRO_HOGAR", "nivel_ed_jefe"]], on=["CODUSU", "NRO_HOGAR"], how="left")
        hogar_t["nivel_ed_bucket_jefe"] = bucket_nivel_ed(hogar_t["nivel_ed_jefe"])

        base = pares.rename(columns={"NRO_HOGAR_t": "NRO_HOGAR"}).merge(
            hogar_t[["CODUSU", "NRO_HOGAR", "AGLOMERADO", "grupo_ingreso", "nivel_ed_bucket_jefe"]],
            on=["CODUSU", "NRO_HOGAR"], how="left",
        ).rename(columns={"NRO_HOGAR": "NRO_HOGAR_t"})
        base["cluster_id"] = base["CODUSU"].astype(str) + "_" + base["AGLOMERADO"].astype(str)
        base = base.merge(ancla_t.rename(columns={"NRO_HOGAR": "NRO_HOGAR_t", **{c: f"{c}_t" for c in ancla_t.columns if c not in ("CODUSU", "NRO_HOGAR")}}),
                           on=["CODUSU", "NRO_HOGAR_t"], how="left")
        base = base.merge(ancla_th.rename(columns={"NRO_HOGAR": "NRO_HOGAR_th", **{c: f"{c}_th" for c in ancla_th.columns if c not in ("CODUSU", "NRO_HOGAR")}}),
                           on=["CODUSU", "NRO_HOGAR_th"], how="left")
        base = base.merge(estrategias_t.assign(CODUSU=hogar_t["CODUSU"], NRO_HOGAR_t=hogar_t["NRO_HOGAR"])[
            ["CODUSU", "NRO_HOGAR_t", "recurre_desahorro", "recurre_endeudamiento_formal"]],
            on=["CODUSU", "NRO_HOGAR_t"], how="left")
        base = base.merge(situacion_t.rename(columns={"NRO_HOGAR": "NRO_HOGAR_t", "situacion_laboral": "situacion_laboral_t"}),
                           on=["CODUSU", "NRO_HOGAR_t"], how="left")
        base = base.merge(situacion_th.rename(columns={"NRO_HOGAR": "NRO_HOGAR_th", "situacion_laboral": "situacion_laboral_th"}),
                           on=["CODUSU", "NRO_HOGAR_th"], how="left")
        base["h"] = h
        partes.append(base)
    cols = ["h", "CODUSU", "AGLOMERADO", "cluster_id", "NRO_HOGAR_t", "NRO_HOGAR_th", "grupo_ingreso", "nivel_ed_bucket_jefe",
            "hacinamiento_nbi_t", "hacinamiento_para_dormir_t", "calidad_materiales_deficitaria_t",
            "condiciones_sanitarias_deficitarias_t", "entorno_deficitario_t", "ancla_con_deficit_t",
            "hacinamiento_nbi_th", "hacinamiento_para_dormir_th", "calidad_materiales_deficitaria_th",
            "condiciones_sanitarias_deficitarias_th", "entorno_deficitario_th", "ancla_con_deficit_th",
            "recurre_desahorro", "recurre_endeudamiento_formal", "situacion_laboral_t", "situacion_laboral_th"]
    return pd.concat(partes, ignore_index=True)[cols] if partes else pd.DataFrame(columns=cols)


t0 = time.time()
base_h1 = construir_base_ancla_condiciones(1)
base_h4 = construir_base_ancla_condiciones(4)
base_ancla_condiciones = pd.concat([base_h1, base_h4], ignore_index=True)
print(f"{len(base_ancla_condiciones):,} pares con identidad confirmada, ancla y situación laboral ({time.time()-t0:.1f}s)")

**2. Estabilidad del ancla**: % de pares donde cambia cada componente entre `t` y
`t+h`, por quintil de origen.

In [ ]:
COMPONENTES_ANCLA = ["hacinamiento_nbi", "hacinamiento_para_dormir", "calidad_materiales_deficitaria",
                      "condiciones_sanitarias_deficitarias", "entorno_deficitario", "ancla_con_deficit"]

filas_estabilidad = []
for h in (1, 4):
    sub_h = base_ancla_condiciones[base_ancla_condiciones["h"] == h]
    for quintil in ORDEN_QUINTILES + [None]:
        sub = sub_h if quintil is None else sub_h[sub_h["grupo_ingreso"] == quintil]
        if sub.empty:
            continue
        fila = {"h": h, "grupo_ingreso": quintil or "TODOS", "n_pares": len(sub)}
        for comp in COMPONENTES_ANCLA:
            cambia = sub[f"{comp}_t"].astype("boolean") != sub[f"{comp}_th"].astype("boolean")
            fila[f"pct_cambia_{comp}"] = round(cambia.mean() * 100, 1)
        filas_estabilidad.append(fila)

ancla_estabilidad = pd.DataFrame(filas_estabilidad)
ancla_estabilidad.to_csv(META / "ancla_habitacional_estabilidad.csv", index=False)
display(ancla_estabilidad)

**Nota (punto 4)**: el cambio es *exactamente* 0.0% en `calidad_materiales_deficitaria`,
`condiciones_sanitarias_deficitarias` y `entorno_deficitario`, en los dos horizontes y
los cinco quintiles -- verificado además con una comprobación directa a nivel de pares
(no es un artefacto del redondeo a 1 decimal: en una muestra de 3.870 pares 2020T2→2020T3,
0 cambiaron `calidad_materiales_deficitaria`). Hipótesis a confirmar contra la
metodología de recolección de INDEC **[inferido, no confirmado contra el manual]**: el
bloque de preguntas de vivienda (`II*`/`IV*`) se releva una sola vez, en la primera
visita a esa vivienda dentro de la ronda de panel, y se arrastra sin volver a
preguntarse en las visitas siguientes -- lo que haría a estos tres componentes
estructuralmente invariantes durante la permanencia de la vivienda en el panel, más
allá de si la condición real cambió o no. `hacinamiento_nbi` sí varía (0.0%-2.5%) porque
depende de `n_miembros`, que sí se releva en cada visita.


**4. `V13`-`V17` (misma pregunta en toda la serie)**: presentes en 86/86 trimestres,
`int64` estable en los 86 -- sin huecos ni cambios de tipo (verificado contra
`data/meta/inventario_variables.csv`). Evidencia indirecta fuerte de continuidad; la
redacción exacta pre-2023T4 no es verificable contra el codebook disponible (solo cubre
2023T4+). La de 2023T4+ (confirmada, codebook p.11) ya está en
`docs/diccionario_variables.md`.

**`NIVEL_ED` antes/después de 2016**: distribución suave y continua 2005→2025 (primario
incompleto 20%→13%, secundario completo 13%→23%, sin instrucción 11%→6%), sin salto en
2016 ni en 2023T4 -- comparable en toda la serie (verificado antes de este plan).

**5-6. Estado de condiciones de vida y transiciones**: 4 estados de situación laboral
del hogar, cruzados con `ancla_con_deficit` (en `t`) y quintil de origen. Celdas con
menos de 30 pares se marcan `[n bajo]`, no se ocultan.

In [ ]:
UMBRAL_N_BAJO = 30


def matriz_situacion_laboral(df):
    n = pd.crosstab(df["situacion_laboral_t"], df["situacion_laboral_th"]).reindex(
        index=ORDEN_SITUACION_LABORAL, columns=ORDEN_SITUACION_LABORAL, fill_value=0)
    total_fila = n.sum(axis=1)
    pct = (n.div(total_fila.replace(0, np.nan), axis=0) * 100).round(1)
    return n, pct, total_fila


filas_transicion_laboral = []
for h in (1, 4):
    sub_h = base_ancla_condiciones[base_ancla_condiciones["h"] == h]
    for quintil in ORDEN_QUINTILES:
        sub_q = sub_h[sub_h["grupo_ingreso"] == quintil]
        for con_deficit, nombre_deficit in [(True, "con_deficit"), (False, "sin_deficit")]:
            sub_qd = sub_q[sub_q["ancla_con_deficit_t"].astype("boolean") == con_deficit]
            for bucket_ed in ["secundario incompleto o menos", "secundario completo o más"]:
                sub_final = sub_qd[sub_qd["nivel_ed_bucket_jefe"] == bucket_ed]
                if sub_final.empty:
                    continue
                n_mat, pct_mat, total_fila = matriz_situacion_laboral(sub_final)
                n_total = len(sub_final)
                marca = "" if n_total >= UMBRAL_N_BAJO else " [n bajo]"
                n_informal_t = int(total_fila.get("solo_asalariados_informales", 0))
                n_formal_t = int(total_fila.get("algun_asalariado_formal", 0))

                fila = {
                    "h": h, "grupo_ingreso": quintil, "ancla": nombre_deficit, "nivel_ed_jefe": bucket_ed,
                    "n_pares": n_total, "n_informal_t": n_informal_t, "n_formal_t": n_formal_t,
                    "marca_n_bajo": marca.strip(),
                }
                # Punto 2: destino desagregado desde "solo_asalariados_informales"
                if n_informal_t > 0:
                    o = "solo_asalariados_informales"
                    fila["informal_pasa_a_formal"] = float(pct_mat.loc[o, "algun_asalariado_formal"])
                    fila["informal_pasa_a_independiente"] = float(pct_mat.loc[o, "algun_independiente_sin_asalariado_formal"])
                    fila["informal_pasa_a_sin_ocupados"] = float(pct_mat.loc[o, "sin_ocupados"])
                    fila["informal_se_queda"] = float(pct_mat.loc[o, o])
                    fila["pct_sale_de_solo_informales"] = round(100 - fila["informal_se_queda"], 1)
                else:
                    fila["informal_pasa_a_formal"] = fila["informal_pasa_a_independiente"] = None
                    fila["informal_pasa_a_sin_ocupados"] = fila["informal_se_queda"] = None
                    fila["pct_sale_de_solo_informales"] = None
                # Punto 2: destino desagregado de la caída desde "algún_asalariado_formal"
                if n_formal_t > 0:
                    o = "algun_asalariado_formal"
                    fila["formal_pasa_a_independiente"] = float(pct_mat.loc[o, "algun_independiente_sin_asalariado_formal"])
                    fila["formal_pasa_a_informal"] = float(pct_mat.loc[o, "solo_asalariados_informales"])
                    fila["formal_pasa_a_sin_ocupados"] = float(pct_mat.loc[o, "sin_ocupados"])
                    fila["formal_se_queda"] = float(pct_mat.loc[o, o])
                    fila["pct_cae_desde_formal"] = round(100 - fila["formal_se_queda"], 1)
                else:
                    fila["formal_pasa_a_independiente"] = fila["formal_pasa_a_informal"] = None
                    fila["formal_pasa_a_sin_ocupados"] = fila["formal_se_queda"] = None
                    fila["pct_cae_desde_formal"] = None
                filas_transicion_laboral.append(fila)

condiciones_vida_transiciones = pd.DataFrame(filas_transicion_laboral)
condiciones_vida_transiciones.to_csv(META / "condiciones_vida_transiciones.csv", index=False)
print(f"Tabla completa ({len(condiciones_vida_transiciones)} filas, quintil x déficit x nivel educativo x horizonte,")
print("con destino desagregado del punto 2) en data/meta/condiciones_vida_transiciones.csv")
display(condiciones_vida_transiciones)


**1. Resumen con/sin déficit, estandarizado por composición** (reemplaza la
comparación anterior). La comparación cruda de tasas con/sin déficit mezcla el efecto
del déficit con la composición: los hogares con déficit están más concentrados en
quintiles bajos y en el bucket educativo más bajo, y esos estratos ya tienen tasas
distintas de por sí. Para aislar el efecto de composición, se calcula la
**estandarización directa**: la tasa esperada de los hogares con déficit *si tuvieran
las tasas de los hogares sin déficit*, tomando la tasa sin-déficit específica de cada
celda quintil x educación, y ponderándolas por la distribución de origen de los propios
hogares con déficit en esas celdas (no por la distribución de los sin-déficit). La
diferencia entre la tasa observada con déficit y esta tasa estandarizada es la parte del
gap que *no* se explica por composición.

**Corrección a la nota anterior**: la vez pasada, (a) "todos los pares" y (b) "ponderado
por n dentro de quintil x educación" daban exactamente el mismo número, y lo atribuí a
que el código `NIVEL_ED` 9 (Ns/Nr) no aparece nunca en los datos. Eso es cierto pero no
es la causa real: (b) estaba filtrando solo por nivel educativo conocido, sin replicar
también el filtro de quintil conocido (Q1-Q5) que sí usan las celdas de la tabla de
arriba -- con ese filtro faltante, (b) terminaba siendo *algebraicamente* el mismo
conjunto de pares que (a) (el filtro de educación no sacaba a nadie), sin que hiciera
falta invocar el código 9 para explicarlo. La estandarización de abajo sí restringe
ambos lados (con y sin déficit) a quintil Q1-Q5 y bucket educativo conocidos -- es el
universo real de la tabla de celdas.

**Punto 2 (eventos del resumen)**: se reemplaza "sale de solo informales" (cualquier
salida) por el evento más específico "pasa de solo informales a algún formal", y se
agrega "pasa a sin ocupados" como destino propio, desde informales y desde formales. Se
mantiene "cae desde algún formal" (cualquier salida de formal, igual definición que
antes).

**Punto 3 (clusters)**: el IC 95% usa bootstrap agrupado por `cluster_id` =
`CODUSU + AGLOMERADO` en vez de `CODUSU` solo -- el `CODUSU` numérico de la era
histórica (pre-2016) no es necesariamente único entre aglomerados distintos.


In [ ]:
EVENTOS_RESUMEN = [
    ("solo_asalariados_informales", "informal_pasa_a_algun_formal",
     lambda d: d["situacion_laboral_th"] == "algun_asalariado_formal"),
    ("solo_asalariados_informales", "informal_pasa_a_sin_ocupados",
     lambda d: d["situacion_laboral_th"] == "sin_ocupados"),
    ("algun_asalariado_formal", "formal_pasa_a_sin_ocupados",
     lambda d: d["situacion_laboral_th"] == "sin_ocupados"),
    ("algun_asalariado_formal", "cae_desde_algun_formal",
     lambda d: d["situacion_laboral_th"] != "algun_asalariado_formal"),
]


def _tabla_cluster(sub_origin, es_evento_fn):
    """Tabla ancha por cluster_id: para cada (con/sin déficit, estrato quintil x
    educación) guarda n de pares y n de eventos -- insumo tanto del estadístico real
    como de cada réplica bootstrap (ver más abajo)."""
    d = sub_origin.copy()
    d["evento"] = es_evento_fn(d)
    d["ancla_lbl"] = np.where(d["ancla_con_deficit_t"].astype("boolean"), "con", "sin")
    d["estrato"] = d["grupo_ingreso"].astype(str) + "__" + d["nivel_ed_bucket_jefe"].astype(str)
    grp = d.groupby(["cluster_id", "ancla_lbl", "estrato"])["evento"].agg(n="size", s="sum").reset_index()
    piv_n = grp.pivot_table(index="cluster_id", columns=["ancla_lbl", "estrato"], values="n", fill_value=0, aggfunc="sum")
    piv_s = grp.pivot_table(index="cluster_id", columns=["ancla_lbl", "estrato"], values="s", fill_value=0, aggfunc="sum")
    return piv_n.values.astype("float64"), piv_s.values.astype("float64"), list(piv_n.columns)


def _estandarizar(N, S, cols):
    """De la tabla ancha (clusters ya sumados, real o de una réplica bootstrap) calcula
    (pct_observado_con_deficit, pct_estandarizado_sin_deficit) -- ver nota del punto 1."""
    idx_con = [i for i, (a, _) in enumerate(cols) if a == "con"]
    estratos_con = {e: i for i, (a, e) in enumerate(cols) if a == "con"}
    estratos_sin = {e: i for i, (a, e) in enumerate(cols) if a == "sin"}
    comunes = sorted(set(estratos_con) & set(estratos_sin))
    idx_con_e = np.array([estratos_con[e] for e in comunes])
    idx_sin_e = np.array([estratos_sin[e] for e in comunes])

    n_con_tot = N[..., idx_con].sum(axis=-1)
    s_con_tot = S[..., idx_con].sum(axis=-1)
    pct_obs = np.where(n_con_tot > 0, s_con_tot / np.where(n_con_tot > 0, n_con_tot, 1) * 100, np.nan)

    n_con_e = N[..., idx_con_e]
    n_sin_e = N[..., idx_sin_e]
    s_sin_e = S[..., idx_sin_e]
    tasa_sin_e = np.divide(s_sin_e, n_sin_e, out=np.full_like(s_sin_e, np.nan), where=n_sin_e > 0)
    peso_e = np.where(np.isnan(tasa_sin_e), 0.0, n_con_e)
    suma_peso = peso_e.sum(axis=-1)
    pct_std = np.where(suma_peso > 0,
                        np.nansum(peso_e * np.nan_to_num(tasa_sin_e), axis=-1) / np.where(suma_peso > 0, suma_peso, 1) * 100,
                        np.nan)
    return pct_obs, pct_std, n_con_tot


def resumen_estandarizado(sub_origin, es_evento_fn, n_boot=400, seed=42, batch=100):
    """Estandarización directa + IC 95% de la diferencia por bootstrap de clusters
    (`cluster_id`, punto 3) -- pesos Poisson(1) i.i.d. por cluster, equivalente
    asintótico al bootstrap multinomial estándar para el número de clusters en juego
    (miles a cientos de miles) y mucho más liviano de vectorizar."""
    N, S, cols = _tabla_cluster(sub_origin, es_evento_fn)
    n_clusters = N.shape[0]
    N_tot, S_tot = N.sum(axis=0, keepdims=True), S.sum(axis=0, keepdims=True)
    pct_obs_arr, pct_std_arr, n_con_tot_arr = _estandarizar(N_tot, S_tot, cols)
    pct_obs, pct_std, n_con_tot = pct_obs_arr[0], pct_std_arr[0], n_con_tot_arr[0]
    diferencia = pct_obs - pct_std
    n_viviendas_con = int((N[..., [i for i, (a, _) in enumerate(cols) if a == "con"]].sum(axis=-1) > 0).sum())

    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    hecho = 0
    while hecho < n_boot:
        b = min(batch, n_boot - hecho)
        W = rng.poisson(1.0, size=(b, n_clusters)).astype("float64")
        Nb, Sb = W @ N, W @ S
        pct_obs_b, pct_std_b, _ = _estandarizar(Nb, Sb, cols)
        diffs[hecho:hecho + b] = pct_obs_b - pct_std_b
        hecho += b
    diffs = diffs[~np.isnan(diffs)]
    ic_low, ic_high = (float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))) if len(diffs) else (np.nan, np.nan)

    return {
        "pct_observado_con_deficit": round(float(pct_obs), 1), "pct_estandarizado_sin_deficit": round(float(pct_std), 1),
        "diferencia": round(float(diferencia), 1),
        "diferencia_ic95_low": round(ic_low, 1) if pd.notna(ic_low) else np.nan,
        "diferencia_ic95_high": round(ic_high, 1) if pd.notna(ic_high) else np.nan,
        "n_pares_con_deficit": int(n_con_tot), "n_viviendas_con_deficit": n_viviendas_con,
        "n_boot_validos": int(len(diffs)),
    }


filas_resumen_deficit = []
for h in (1, 4):
    sub_h = base_ancla_condiciones[
        (base_ancla_condiciones["h"] == h)
        & base_ancla_condiciones["grupo_ingreso"].isin(ORDEN_QUINTILES)
        & base_ancla_condiciones["nivel_ed_bucket_jefe"].notna()
    ]
    for origen, evento, es_evento_fn in EVENTOS_RESUMEN:
        sub_origin = sub_h[sub_h["situacion_laboral_t"] == origen]
        if sub_origin.empty:
            continue
        fila = {"h": h, "evento": evento}
        fila.update(resumen_estandarizado(sub_origin, es_evento_fn))
        filas_resumen_deficit.append(fila)

resumen_deficit = pd.DataFrame(filas_resumen_deficit)
resumen_deficit.to_csv(META / "condiciones_vida_resumen_deficit.csv", index=False)
print("--- Resumen con/sin déficit, estandarizado por quintil x educación, IC 95% bootstrap por vivienda ---")
display(resumen_deficit)


Todo Parte E es descriptivo -- nada de lo anterior se interpreta como causal.